In [2]:
import os
%pwd
os.chdir("../")
from dataclasses import dataclass
from pathlib import Path
os.getcwd()

'/home/michauj/ds_e2e_project'

In [3]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/MichauJ/ds_e2e_project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="MichauJ"
os.environ["MLFLOW_TRACKING_PASSWORD"]="c5474382a67159587598e99dc3ea53aac3e6f124"


In [7]:
@dataclass
class ModelEvaluationConfig:
    root_dir:Path
    test_data_path:Path
    model_path:Path
    all_params:dict
    metric_file_name: Path
    target_column:str
    mlflow_uri:str

In [8]:
from src.ds_e2e_project.constants import *
from src.ds_e2e_project.utils.common import read_yaml, create_directories,save_json

In [9]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath = PARAMS_FILE_PATH,schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
        root_dir = config.root_dir,
        test_data_path = config.test_data_path,
        model_path = config.model_path,
        mlflow_uri = config.mlflow_uri,
        all_params=params,
        metric_file_name = config.metric_file_name,
        target_column = schema.name
        )

        return model_evaluation_config

In [11]:
import os
import pandas as pd
from sklearn.metrics import root_mean_squared_error,mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [17]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    def eval_metrics(self, actual, pred):
        rmse, mae, r2 = (f(actual, pred) for f in (root_mean_squared_error, mean_absolute_error, r2_score))
        return rmse, mae, r2
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        test_x = test_data.drop([self.config.target_column], axis = 1)
        test_y = test_data[[self.config.target_column]]
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            rmse, mae, r2 = self.eval_metrics(test_y,predicted_qualities)
            scores = {"rmse":rmse,"mae":mae,"r2":r2}
            save_json(path=Path(self.config.metric_file_name),data = scores)
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({
    "rmse": rmse,
    "mae": mae,
    "r2": r2
})
        if tracking_url_type_store != "file":
            mlflow.sklearn.log_model(model,"model",registered_model_name="ElasticnetModel")
        else:
            mlflow.sklearn.log_model(model,"model")

In [18]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config = model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-01 16:18:44,566: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-01 16:18:44,568: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-01 16:18:44,571: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-01 16:18:44,572: INFO: common: created directory at artifacts]
[2026-05-01 16:18:44,573: INFO: common: created directory at artifacts/model_evaluation]
[2026-05-01 16:18:45,818: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]
🏃 View run marvelous-newt-287 at: https://dagshub.com/MichauJ/ds_e2e_project.mlflow/#/experiments/0/runs/2d86a749c1614911b6166a9a23b61f58
🧪 View experiment at: https://dagshub.com/MichauJ/ds_e2e_project.mlflow/#/experiments/0


2026/05/01 16:18:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 16:18:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/michauj/ds_e2e_project
2026/05/01 16:18:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/01 16:18:47 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/michauj/ds_e2e_project
2026/05/01 16:18:47 INFO mlflow.utils.environment: Detected uv project at /home/michauj/ds_e2e_project. Attempting to export requirements via 'uv export'.
2026/05/01 16:18:47 INFO mlflow.utils.uv_utils: Exported 189 dependen